# Full nonparagraph chunk review

Review the full frozen 16,540-parent nonparagraph scope. Reuse the verified sample
and resolve the 204 semantic escalations and 112 reference-budget failures from
Notebook 09. Native Codex desktop windows review assigned local text. PySpark
preserves the original parent text and inherited split on every candidate plan.

## Reconcile the earlier sample and legacy blockers

Read the verified Notebook 09 receipts and prepare immutable case assignments. The case roster preserves the original parent and source-packet bindings.

In [ ]:
import hashlib
import json
import os
from collections import Counter
from pathlib import Path

workspace = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/09_targeted_sample_review.ipynb").is_file()
)
root = workspace / ".cache/chunk-policy-review"
previous = root / "full-sample-v1"
run = root / "full-nonparagraph-v1"
case_root = run / "case_review"
case_root.mkdir(mode=0o700, parents=True, exist_ok=True)
(case_root / "decisions").mkdir(mode=0o700, exist_ok=True)


def get(path):
    return json.loads(path.read_text(encoding="utf-8"))


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def publish(path, data):
    encoded = json.dumps(data, ensure_ascii=False, indent=2) + "\n"
    if path.exists():
        if path.read_text(encoding="utf-8") != encoded:
            raise ValueError(f"Existing immutable file changed: {path}.")
    else:
        fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o400)
        with os.fdopen(fd, "w", encoding="utf-8") as handle:
            handle.write(encoded)


parents = {}
origins = {}
for key, directory, name in (
    ("diagnostic", root / "96e814e5c00fefa5a9ef", "TRIAL_REPLAY.json"),
    ("sample-001", root / "3d89641f2a281193dc7a", "BATCH_REPLAY.json"),
):
    replay = directory / name
    record = get(replay)
    for parent in record["parents"]:
        identifier = parent["release_row"]
        if identifier in parents:
            raise ValueError("Earlier sample parents overlap.")
        parents[identifier] = parent
        origins[identifier] = key
    for batch, digest in record["receipts"].items():
        if sha(directory / "decisions" / f"{batch}.jsonl") != digest:
            raise ValueError("An earlier diagnostic receipt changed.")
manifest = get(previous / "MANIFEST.json")
for batch, item in manifest["batches"].items():
    receipt = previous / "decisions" / f"{batch}.jsonl"
    result = get(previous / "replays" / f"{batch}.json")
    if (
        result["receipt_sha256"] != sha(receipt)
        or result["assignment_sha256"] != item["sha256"]
        or sha(previous / item["file"]) != item["sha256"]
    ):
        raise ValueError(f"The prior sample candidate no longer matches {batch}.")
    for parent in result["parents"]:
        identifier = parent["release_row"]
        if identifier in parents:
            raise ValueError(f"Repeated sample membership: {identifier}.")
        parents[identifier] = parent
        origins[identifier] = batch
if len(parents) != 4635:
    raise ValueError("All 4,635 parent replays are required.")
first_adjudication = root / "3d89641f2a281193dc7a" / "adjudication" / "ADJUDICATION_REPLAY.json"
for decision in get(first_adjudication)["adjudications"]:
    row = parents[decision["release_row"]]
    disposition = decision["verified_disposition"]
    if disposition == "retain_reference":
        row["status"] = "boundary_review_complete"
        row["internal_end_offsets"] = None
    elif disposition == "repair_reference":
        row["status"] = "boundary_review_complete"
        row["internal_end_offsets"] = decision["replacement_ends"][:-1]
    elif disposition == "hold_for_model_input":
        row["status"] = "hold_for_model_input"
    else:
        raise ValueError("The initial five adjudications have an open disposition.")
    row["adjudication_receipt_sha256"] = decision["receipt_sha256"]

statuses = Counter(row["status"] for row in parents.values())
that_closure = previous / "SAMPLE_CLOSURE.json"
if dict(statuses) != get(that_closure)["current_dispositions"]:
    raise ValueError("The prior closure's parent dispositions do not reconcile.")
prior_cases = {
    key: {**row, "review_origin": origins[key]}
    for key, row in parents.items()
    if row["status"] in {"escalated", "reference_budget_review_required"}
}
if Counter(row["status"] for row in prior_cases.values()) != {
    "escalated": 204,
    "reference_budget_review_required": 112,
}:
    raise ValueError("The original blocker roster did not reconcile to 204 and 112.")

source_bundles = {
    "diagnostic": root / "96e814e5c00fefa5a9ef",
    "sample-001": root / "3d89641f2a281193dc7a",
}
for key, case in prior_cases.items():
    origin = origins[key]
    folder = source_bundles.get(origin, previous)
    files = (
        sorted(folder.glob("BATCH_00?.json"))
        if origin in source_bundles
        else [folder / f"{origin}.json"]
    )
    entries = []
    for f in files:
        assignment = get(f)
        selected = {u["packet_sha256"]: u for u in assignment["units"] if u["release_row"] == key}
        if not selected:
            continue
        final = folder / assignment["receipt"]
        for item in map(json.loads, final.read_text(encoding="utf-8").splitlines()):
            if item["packet_sha256"] in selected:
                u = selected[item["packet_sha256"]]
                entries.append(
                    {
                        "packet_line": u["line"],
                        "target_boundary_ids": u["target_boundary_ids"],
                        "decision": item,
                    }
                )
    if not entries:
        raise ValueError(f"No earlier semantic evidence for {key}.")
    case["previous_packets"] = str((folder / "PACKETS.jsonl").resolve())
    case["previous_decisions"] = entries

# Round-robin across each risk group so each assignment contains complete parents.
items = sorted(prior_cases.values(), key=lambda row: (row["status"], row["release_row"]))
assignments = {}
for i, offset in enumerate(range(0, len(items), 12), start=1):
    label = f"prior-case-{i:03d}"
    file = case_root / f"{label}.json"
    roster = items[offset : offset + 12]
    publish(
        file,
        {
            "batch": label,
            "cases": roster,
            "receipt": f"decisions/{label}.jsonl",
            "sample_closure_sha256": sha(that_closure),
        },
    )
    assignments[label] = {"file": file.name, "parents": len(roster), "sha256": sha(file)}

prior_status = run / "SAMPLE_PARENT_DISPOSITIONS.json"
publish(
    prior_status,
    {
        "sample_closure_sha256": sha(that_closure),
        "parents": sorted(parents.values(), key=lambda row: row["release_row"]),
    },
)
publish(
    case_root / "MANIFEST.json",
    {
        "sample_closure_sha256": sha(that_closure),
        "parents": sorted(prior_cases),
        "initial_status_counts": dict(Counter(row["status"] for row in prior_cases.values())),
        "batches": assignments,
    },
)
print(
    json.dumps(
        {"cases": len(prior_cases), "assignments": len(assignments), "prior": sha(prior_status)}
    )
)


## Bind the corpus, frozen sample and earlier cases

Read only the verified targeted scope, reference plan, and earlier sample dispositions.

In [ ]:
import json
import tempfile
from pathlib import Path

import nbformat
import pandas as pd
from IPython.display import display
from pyspark.sql import Row
from pyspark.sql import functions as F

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/08_targeted_chunk_review.ipynb").is_file()
)
n08 = nbformat.read(ROOT / "notebooks/08_targeted_chunk_review.ipynb", as_version=4)
cell = next(item for item in n08.cells if item.id == "08-bind")
upstream08 = {"__name__": "__main__"}
exec(compile(cell.source, "08_targeted_chunk_review.ipynb:08-bind", "exec"), upstream08)
parents = upstream08["parents"]
plans = upstream08["plans"]
spark = upstream08["spark"]
sha256_file = upstream08["sha256_file"]
canonical_sha256 = upstream08["canonical_sha256"]
make_review_packets = upstream08["make_review_packets"]
encoding = upstream08["encoding"]
budget = upstream08["budget"]
CACHE = ROOT / ".cache/chunk-policy-review"
RUN = CACHE / "full-nonparagraph-v1"
POLICY_DIR = upstream08["output"]
SAMPLE = CACHE / "full-sample-v1"

policy_manifest = json.loads((POLICY_DIR / "MANIFEST.json").read_text(encoding="utf-8"))
policy_path = POLICY_DIR / "CHUNK_POLICY_V1_CANDIDATE.json"
if sha256_file(policy_path) != policy_manifest["policy_sha256"]:
    raise ValueError("The targeted policy receipt changed.")
policy = json.loads(policy_path.read_text(encoding="utf-8"))
if policy["reference_manifest_sha256"] != sha256_file(
    upstream08["reference_dir"] / "MANIFEST.json"
):
    raise ValueError("The targeted policy belongs to a different reference plan.")
load_table = upstream08["upstream"]["bound_review_table"]
risk = load_table(POLICY_DIR, "nonparagraph_scope", policy_manifest["tables"]["nonparagraph_scope"])
sample = load_table(POLICY_DIR, "semantic_sample", policy_manifest["tables"]["semantic_sample"])
risk_size = risk.count()
sample_closure_path = SAMPLE / "SAMPLE_CLOSURE.json"
sample_closure_sha = sha256_file(sample_closure_path)
sample_closure = json.loads(sample_closure_path.read_text(encoding="utf-8"))
prior_path = RUN / "SAMPLE_PARENT_DISPOSITIONS.json"
prior_document = json.loads(prior_path.read_text(encoding="utf-8"))
case_manifest_path = RUN / "case_review/MANIFEST.json"
case_manifest = json.loads(case_manifest_path.read_text(encoding="utf-8"))
if (
    risk_size != 16540
    or sample_closure["replayed_parents"] != 4635
    or sample_closure_sha != prior_document["sample_closure_sha256"]
    or sample_closure_sha != case_manifest["sample_closure_sha256"]
):
    raise ValueError("The full review must bind the verified risk scope and sample closure.")
prior = {row["release_row"]: row for row in prior_document["parents"]}
sample_rows = {
    row.release_row: row
    for row in sample.select("release_row", "clean_sha256", "sample", "ml_split").collect()
}
if (
    len(prior) != 4635
    or set(prior) != set(sample_rows)
    or any(
        row["body_sha256"] != sample_rows[key].clean_sha256
        or row["sample"] != sample_rows[key].sample
        or row["ml_split"] != sample_rows[key].ml_split
        for key, row in prior.items()
    )
):
    raise ValueError("The prior decisions differ from the verified sample roster.")
pending = {
    key
    for key, row in prior.items()
    if row["status"] in {"escalated", "reference_budget_review_required"}
}
if pending != set(case_manifest["parents"]) or len(pending) != 316:
    raise ValueError("The earlier blocker identities did not reconcile.")
first_pass = {key: row for key, row in prior.items() if row["sample"] == "nonparagraph"}
if len(first_pass) != 4135:
    raise ValueError("The targeted sample overlap did not reconcile.")


## Publish whole-parent nonparagraph review batches

Register the 4,135 prior nonparagraph sample reviews and stage the remaining parents. Use the same packet and evidence contracts as Notebook 09.

In [ ]:
from collections import Counter

SHARED = RUN / "review"
core = upstream08["CODEPOINT_WINDOW"]
margin = upstream08["CONTEXT_WIDTH"]
batch_target = 48
identity = {
    "policy": "full_nonparagraph_review_v1",
    "reference_manifest_sha256": policy["reference_manifest_sha256"],
    "targeted_scope_manifest_sha256": sha256_file(POLICY_DIR / "MANIFEST.json"),
    "sample_closure_sha256": sample_closure_sha,
    "prior_dispositions_sha256": sha256_file(prior_path),
    "legacy_adjudication_manifest_sha256": sha256_file(case_manifest_path),
}
if not SHARED.exists():
    pending_scope = risk.where(~F.col("release_row").isin(*sorted(first_pass)))
    selected = (
        pending_scope.select(
            "release_row",
            "clean_sha256",
            "clean_chars",
            "risk",
            "format",
            "ml_split",
            "nonparagraph_offsets",
            "target_windows",
        )
        .join(parents.select("release_row", "clean_text", "paragraphs"), "release_row")
        .join(plans.select("release_row", "plan"), "release_row")
        .orderBy(F.sha2(F.col("release_row").cast("string"), 256), "release_row")
    )
    with tempfile.TemporaryDirectory(prefix=".full-review-", dir=RUN) as temp:
        stage = Path(temp) / "snapshot"
        stage.mkdir(mode=0o700)
        (stage / "decisions").mkdir(mode=0o700)
        (stage / "dispatch").mkdir(mode=0o700)
        batches = {}
        items = []
        observed = set()
        windows_by_split = Counter()
        line = 0

        def publish_batch(units):
            label = f"nonparagraph-{len(batches) + 1:04d}"
            path = stage / f"{label}.json"
            data = {"batch": label, "units": units, "receipt": f"decisions/{label}.jsonl"}
            path.write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
            path.chmod(0o400)
            batches[label] = {
                "file": path.name,
                "sha256": sha256_file(path),
                "windows": len(units),
                "parents": len({u["release_row"] for u in units}),
            }

        packet_path = stage / "PACKETS.jsonl"
        with packet_path.open("x", encoding="utf-8") as handle:
            for row in selected.toLocalIterator():
                if row.release_row in observed or row.release_row in first_pass:
                    raise ValueError("The full review roster repeats a previously assigned parent.")
                observed.add(row.release_row)
                geometry = [
                    Row(
                        unit_ordinal=i,
                        start=start,
                        end=min(start + core, row.clean_chars),
                        context_start=max(0, start - margin),
                        context_end=min(start + core + margin, row.clean_chars),
                    )
                    for i, start in enumerate(range(0, row.clean_chars, core))
                ]
                binding = Row(
                    release_row=row.release_row,
                    clean_sha256=row.clean_sha256,
                    clean_chars=row.clean_chars,
                    plan=row.plan,
                    plan_sha256=canonical_sha256(
                        {
                            "reference_manifest_sha256": policy["reference_manifest_sha256"],
                            "body_sha256": row.clean_sha256,
                            "ends": row.plan.ends,
                        }
                    ),
                    review_units=geometry,
                )
                target_windows = set(row.target_windows)
                selected_packets = [
                    packet
                    for packet in make_review_packets(binding, row.clean_text, row.paragraphs)
                    if packet["unit_ordinal"] in target_windows
                ]
                if not selected_packets or len(selected_packets) != len(target_windows):
                    raise ValueError("A full-review parent lost a targeted work window.")
                if items and len(items) + len(selected_packets) > batch_target:
                    publish_batch(items)
                    items = []
                for packet in selected_packets:
                    target_ids = [
                        end
                        for end in row.nonparagraph_offsets
                        if packet["start"] <= end < packet["end"]
                    ]
                    if not target_ids or not set(target_ids) <= set(
                        packet["baseline_boundary_ids"]
                    ):
                        raise ValueError(
                            "A referenced nonparagraph edge is missing from its packet."
                        )
                    line += 1
                    handle.write(json.dumps(packet, ensure_ascii=False) + "\n")
                    items.append(
                        {
                            "line": line,
                            "release_row": row.release_row,
                            "risk": row.risk,
                            "format": row.format,
                            "ml_split": row.ml_split,
                            "packet_sha256": packet["packet_sha256"],
                            "target_boundary_ids": target_ids,
                        }
                    )
                windows_by_split[row.ml_split] += len(selected_packets)
        if items:
            publish_batch(items)
        expected = {row.release_row for row in risk.select("release_row").collect()}
        if observed.isdisjoint(first_pass) and observed.union(first_pass) == expected:
            pass
        else:
            raise ValueError("The full review assignments do not cover the 16,540-parent scope.")
        packet_path.chmod(0o400)
        manifest = {
            **identity,
            "packet_sha256": sha256_file(packet_path),
            "baseline_sample_parents": len(first_pass),
            "remaining_parents": len(observed),
            "remaining_target_windows": line,
            "windows_by_split": dict(windows_by_split),
            "batches": batches,
        }
        m = stage / "MANIFEST.json"
        m.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
        m.chmod(0o400)
        if SHARED.exists():
            raise FileExistsError("The full review snapshot appeared during packet preparation.")
        stage.rename(SHARED)

manifest_path = SHARED / "MANIFEST.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if any(manifest.get(key) != value for key, value in identity.items()):
    raise ValueError("The saved full review run belongs to another scope.")
if sha256_file(SHARED / "PACKETS.jsonl") != manifest["packet_sha256"]:
    raise ValueError("The frozen full-review packet file changed.")
assignments = {}
parent_roster = set(first_pass)
line_roster = set()
for label, item in manifest["batches"].items():
    path = SHARED / item["file"]
    if sha256_file(path) != item["sha256"]:
        raise ValueError(f"The assignment {label} changed.")
    assignment = json.loads(path.read_text(encoding="utf-8"))
    members = {row["release_row"] for row in assignment["units"]}
    lines = {row["line"] for row in assignment["units"]}
    if (
        assignment["batch"] != label
        or assignment["receipt"] != f"decisions/{label}.jsonl"
        or len(lines) != len(assignment["units"])
        or len(lines) != item["windows"]
        or len(members) != item["parents"]
        or members & parent_roster
        or lines & line_roster
    ):
        raise ValueError(f"Invalid review roster for {label}.")
    assignments[label] = assignment
    parent_roster.update(members)
    line_roster.update(lines)
if len(parent_roster) != risk_size or line_roster != set(
    range(1, manifest["remaining_target_windows"] + 1)
):
    raise ValueError("The full review membership could not be reconciled.")
summary = {
    "sample_reviews_reused": len(first_pass),
    "remaining_parents": manifest["remaining_parents"],
    "remaining_target_windows": manifest["remaining_target_windows"],
    "new_review_batches": len(assignments),
    "prior_blockers": len(pending),
}
print(summary)
print(f"Shared packet run: {SHARED.relative_to(ROOT)}")


## Bind reviewer lanes and verify the parent roster

Keep one writer for each batch and preserve checkpointed decisions when resuming.

In [ ]:
case_names = sorted(case_manifest["batches"])
review_names = sorted(manifest["batches"])
dispatch = SHARED / "dispatch"
descriptions = []
claimed_review = set()
claimed_cases = set()
for slot in range(3):
    name = f"full-review-lane-{slot + 1:03d}"
    jobs = []
    case_index = 0
    cases = case_names[slot::3]
    for index, label in enumerate(review_names[slot::3]):
        if index % 3 == 0 and case_index < len(cases):
            case = cases[case_index]
            jobs.append({"batch": case, "kind": "adjudication", **case_manifest["batches"][case]})
            case_index += 1
        jobs.append({"batch": label, "kind": "review", **manifest["batches"][label]})
    if case_index != len(cases):
        raise ValueError("A legacy blocker assignment did not receive a lane.")
    lane = {
        "lane": name,
        "review_manifest_sha256": sha256_file(manifest_path),
        "case_manifest_sha256": sha256_file(case_manifest_path),
        "packet_sha256": manifest["packet_sha256"],
        "jobs": jobs,
    }
    path = dispatch / f"{name}.json"
    if path.is_file():
        if json.loads(path.read_text(encoding="utf-8")) != lane:
            raise ValueError(f"The immutable lane changed: {name}.")
    else:
        with path.open("x", encoding="utf-8") as handle:
            handle.write(json.dumps(lane, indent=2) + "\n")
        path.chmod(0o400)
    review_ids = {job["batch"] for job in jobs if job["kind"] == "review"}
    case_ids = {job["batch"] for job in jobs if job["kind"] == "adjudication"}
    if claimed_review & review_ids or claimed_cases & case_ids:
        raise ValueError("The reviewer lanes overlap.")
    claimed_review.update(review_ids)
    claimed_cases.update(case_ids)
    descriptions.append(
        {
            "Lane": name,
            "Review batches": len(review_ids),
            "Prior case batches": len(case_ids),
            "Review windows": sum(manifest["batches"][item]["windows"] for item in review_ids),
        }
    )
if claimed_review != set(review_names) or claimed_cases != set(case_names):
    raise ValueError("The reviewer lanes omit assignments.")
display(pd.DataFrame(descriptions))


## Verify the six-window handoff

Check the replacement ownership roster and reconcile preserved checkpoint rows after the assigned receipts arrive.

In [ ]:
for wave in (2, 3):
    handoff_path = dispatch / f"WAVE{wave}_HANDOFF.json"
    if not handoff_path.is_file():
        continue
    handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
    if handoff["review_manifest_sha256"] != sha256_file(manifest_path):
        raise ValueError("The six-window handoff refers to a different review run.")
    all_jobs = set()
    for label, metadata in handoff["lanes"].items():
        lane_path = dispatch / metadata["file"]
        lane = json.loads(lane_path.read_text(encoding="utf-8"))
        labels = [job["batch"] for job in lane["jobs"]]
        if (
            sha256_file(lane_path) != metadata["sha256"]
            or lane["lane"] != label
            or lane["packet_sha256"] != manifest["packet_sha256"]
            or len(labels) != metadata["batches"]
            or len(labels) != len(set(labels))
            or not set(labels) <= set(assignments)
            or all_jobs.intersection(labels)
        ):
            raise ValueError("The redistributed wave contains an invalid lease.")
        all_jobs.update(labels)
    if len(all_jobs) != handoff["unfinished_batch_count"]:
        raise ValueError("The wave two assignment count differs from the handoff roster.")
    dispatch_record = json.loads(
        (dispatch / f"WAVE{wave}_DISPATCH.json").read_text(encoding="utf-8")
    )
    owners = {item["threadId"]: item["lane"] for item in dispatch_record["bindings"]}
    if len(owners) != 6 or set(owners.values()) != set(handoff["lanes"]):
        raise ValueError("The handoff does not have six distinct writers.")
    saved = 0
    checked = 0
    for label, binding in handoff["protected_checkpoints"].items():
        path = dispatch / binding["file"]
        if sha256_file(path) != binding["sha256"]:
            raise ValueError("An existing reviewer checkpoint changed.")
        rows = path.read_bytes().splitlines()
        if len(rows) != binding["windows"]:
            raise ValueError("An existing reviewer checkpoint lost rows.")
        saved += len(rows)
        final = SHARED / assignments[label]["receipt"]
        if not final.is_file():
            continue
        received = {
            json.loads(line)["packet_sha256"]: line for line in final.read_bytes().splitlines()
        }
        if any(received.get(json.loads(line)["packet_sha256"]) != line for line in rows):
            raise ValueError("A replacement receipt changed a completed checkpoint decision.")
        checked += len(rows)
    print(f"Wave {wave}: verified six lanes and {checked} of {saved} carried-forward decisions.")


## Replay first-pass review receipts

Recalculate the reference token budget and preserve every undispatched boundary.

In [ ]:
import tiktoken

encoder = tiktoken.get_encoding(encoding)
registered = first_pass
sample_rows = {
    row.release_row: row
    for row in risk.select(
        "release_row",
        "clean_sha256",
        "risk",
        "ml_split",
        F.lit("nonparagraph").alias("sample"),
    ).collect()
}

replay_dir = SHARED / "replays"
replay_dir.mkdir(mode=0o700, exist_ok=True)
reviewed = dict(registered)
replay_counts = Counter()
pending_replay = []

for batch, assignment in assignments.items():
    receipt = SHARED / assignment["receipt"]
    replay_file = replay_dir / f"{batch}.json"
    if not receipt.is_file():
        if replay_file.is_file():
            raise ValueError(f"The checked receipt disappeared for {batch}.")
        continue
    receipt_sha = sha256_file(receipt)
    assignment_sha = manifest["batches"][batch]["sha256"]
    if replay_file.is_file():
        saved = json.loads(replay_file.read_text(encoding="utf-8"))
        if (saved["receipt_sha256"], saved["assignment_sha256"]) != (receipt_sha, assignment_sha):
            raise ValueError(f"The checked inputs changed for {batch}.")
        for row in saved["parents"]:
            if row["release_row"] in reviewed:
                raise ValueError("Two replays contain the same sampled parent.")
            reviewed[row["release_row"]] = row
        replay_counts.update(saved["decision_counts"])
    else:
        pending_replay.append((batch, receipt, receipt_sha, assignment_sha))

if pending_replay:
    assignments_to_replay = {
        batch: {unit["packet_sha256"]: unit for unit in assignments[batch]["units"]}
        for batch, _, _, _ in pending_replay
    }
    target_lines = {
        unit["line"]: digest
        for units in assignments_to_replay.values()
        for digest, unit in units.items()
    }
    packets = {}
    with (SHARED / "PACKETS.jsonl").open(encoding="utf-8") as handle:
        for line, raw in enumerate(handle, start=1):
            if line not in target_lines:
                continue
            packet = json.loads(raw)
            digest = packet.pop("packet_sha256")
            if digest != target_lines[line] or canonical_sha256(packet) != digest:
                raise ValueError("A frozen packet line does not match its assignment.")
            packets[digest] = packet
    if set(packets) != set(target_lines.values()):
        raise ValueError("The selected packet lines could not be replayed.")

    ids = sorted({packet["release_row"] for packet in packets.values()})
    corpus = {
        row.release_row: row
        for row in parents.where(F.col("release_row").isin(*ids))
        .select("release_row", "clean_text")
        .collect()
    }
    reference = {
        row.release_row: row
        for row in plans.where(F.col("release_row").isin(*ids))
        .select("release_row", "clean_sha256", "plan")
        .collect()
    }
    if set(corpus) != set(ids) or set(reference) != set(ids):
        raise ValueError("The full reference parents could not be rebound.")

    for batch, receipt, receipt_sha, assignment_sha in pending_replay:
        expected = assignments_to_replay[batch]
        entries = [json.loads(line) for line in receipt.read_text(encoding="utf-8").splitlines()]
        if len(entries) != len(expected) or {row["packet_sha256"] for row in entries} != set(
            expected
        ):
            raise ValueError(f"The receipt roster is incomplete or duplicated: {batch}.")
        parent_decisions = {}
        counts = Counter()
        for entry in entries:
            digest = entry["packet_sha256"]
            packet = packets[digest]
            assigned = expected[digest]
            chosen = entry["selected_boundary_ids"]
            status = entry["status"]
            evidence = entry["evidence_outcome"]
            baseline = packet["baseline_boundary_ids"]
            if (
                not isinstance(chosen, list)
                or any(type(position) is not int for position in chosen)
                or chosen != sorted(set(chosen))
                or not set(chosen) <= set(packet["candidate_boundary_ids"])
                or any(
                    not isinstance(entry.get(key), str) or not entry[key].strip()
                    for key in ("reason", "reviewer", "reviewer_version")
                )
            ):
                raise ValueError(f"Invalid selected boundaries or provenance in {batch}.")
            if not (
                (
                    status == "accept"
                    and chosen == baseline
                    and evidence in {"none", "boundary_preference"}
                )
                or (
                    status == "adjust"
                    and chosen != baseline
                    and evidence == "material_evidence_break"
                    and set(baseline) - set(chosen) <= set(assigned["target_boundary_ids"])
                )
                or (
                    status == "escalate"
                    and not chosen
                    and evidence in {"material_evidence_break", "uncertain"}
                )
            ):
                raise ValueError(f"The evidence finding disagrees with the decision in {batch}.")
            parent_decisions.setdefault(packet["release_row"], []).append((entry, packet))
            counts[status] += 1
        rows = []
        for parent_id, decisions in sorted(parent_decisions.items()):
            if parent_id in reviewed:
                raise ValueError("Two sample receipts claim the same parent.")
            meta = sample_rows[parent_id]
            ref = reference[parent_id]
            body = corpus[parent_id].clean_text
            baseline = ref.plan.ends
            plan_sha = canonical_sha256(
                {
                    "reference_manifest_sha256": policy["reference_manifest_sha256"],
                    "body_sha256": ref.clean_sha256,
                    "ends": baseline,
                }
            )
            if not all(
                packet["body_sha256"] == meta.clean_sha256 == ref.clean_sha256
                and packet["plan_sha256"] == plan_sha
                and packet["context_text"] == body[packet["context_start"] : packet["context_end"]]
                for _, packet in decisions
            ):
                raise ValueError("A review packet did not match its reference text and plan.")
            ends = set(baseline)
            parent_status = "boundary_review_complete"
            token_counts = None
            if any(entry["status"] == "escalate" for entry, _ in decisions):
                parent_status = "escalated"
            else:
                for entry, packet in decisions:
                    ends.difference_update(packet["baseline_boundary_ids"])
                    ends.update(entry["selected_boundary_ids"])
                ends = sorted(ends)
                if ends[-1:] != [len(body)] or ends[0] <= 0:
                    raise ValueError("A completed parent failed coverage.")
                token_counts = [
                    len(encoder.encode_ordinary(body[start:end]))
                    for start, end in zip([0, *ends[:-1]], ends, strict=True)
                ]
                if any(count < 1 or count > budget for count in token_counts):
                    parent_status = "reference_budget_review_required"
            row = {
                "release_row": parent_id,
                "sample": meta.sample,
                "risk": meta.risk,
                "ml_split": meta.ml_split,
                "body_sha256": meta.clean_sha256,
                "review_origin": batch,
                "status": parent_status,
                "internal_end_offsets": ends[:-1]
                if parent_status == "boundary_review_complete"
                else None,
                "token_counts": token_counts
                if parent_status == "boundary_review_complete"
                else None,
                "evidence_outcomes": sorted({entry["evidence_outcome"] for entry, _ in decisions}),
            }
            rows.append(row)
            reviewed[parent_id] = row
        if sha256_file(receipt) != receipt_sha:
            raise ValueError(f"The receipt changed during replay: {batch}.")
        record = {
            "batch": batch,
            "receipt_sha256": receipt_sha,
            "assignment_sha256": assignment_sha,
            "decision_counts": dict(counts),
            "parents": rows,
        }
        with tempfile.NamedTemporaryFile(
            dir=replay_dir, mode="w", encoding="utf-8", delete=False
        ) as handle:
            handle.write(json.dumps(record, indent=2) + "\n")
            staged = Path(handle.name)
        staged.rename(replay_dir / f"{batch}.json")
        replay_counts.update(counts)

print(f"Replayed {len(reviewed)} of {len(sample_rows)} frozen nonparagraph parents.")
display(pd.DataFrame(reviewed.values()).groupby(["sample", "status"]).size().rename("parents"))


## Resolve the earlier sample blockers and new review failures

Accept only validated reference plans or repairs. Keep explicit model-input holds and their source evidence available for audit.

In [ ]:
case_root = RUN / "case_review"
entries = {}
case_receipts = {}
case_assignment = {}
for label, assignment in case_manifest["batches"].items():
    file = case_root / assignment["file"]
    if sha256_file(file) != assignment["sha256"]:
        raise ValueError(f"The old-case assignment changed: {label}.")
    task = json.loads(file.read_text(encoding="utf-8"))
    case_ids = {item["release_row"] for item in task["cases"]}
    if (
        task["batch"] != label
        or task["sample_closure_sha256"] != sample_closure_sha
        or task["receipt"] != f"decisions/{label}.jsonl"
        or len(case_ids) != assignment["parents"]
        or case_ids.intersection(case_assignment)
    ):
        raise ValueError("Legacy case assignments overlap or refer to another sample.")
    case_assignment.update({key: label for key in case_ids})
    receipt = case_root / task["receipt"]
    if not receipt.is_file():
        continue
    rows = [json.loads(line) for line in receipt.read_text(encoding="utf-8").splitlines()]
    if len(rows) != len(case_ids) or {row["release_row"] for row in rows} != case_ids:
        raise ValueError(f"The final case receipt has the wrong parent roster: {label}.")
    entries.update({row["release_row"]: row for row in rows})
    case_receipts[label] = sha256_file(receipt)
if set(case_assignment) != pending:
    raise ValueError("Legacy adjudication assignments do not cover the original blockers.")

case_results = {}
case_validation_failures = {}
if entries:
    selection = F.col("release_row").isin(*sorted(entries))
    ref_cases = {
        row.release_row: row
        for row in plans.where(selection).select("release_row", "clean_sha256", "plan").collect()
    }
    text_cases = {
        row.release_row: row
        for row in parents.where(selection)
        .select("release_row", "clean_text", "paragraphs")
        .collect()
    }
    if set(ref_cases) != set(entries) or set(text_cases) != set(entries):
        raise ValueError("A legacy blocker receipt cannot be rebound to its original parent.")

    sentence_break = upstream08["upstream"]["SENTENCE_BREAK"]
    whitespace_break = upstream08["upstream"]["WHITESPACE_BREAK"]
    for key, entry in sorted(entries.items()):
        ref = ref_cases[key]
        text = text_cases[key]
        disposition = entry["disposition"]
        ends = entry["replacement_ends"]
        failure = None
        if (
            entry["body_sha256"] != prior[key]["body_sha256"]
            or entry["body_sha256"] != ref.clean_sha256
            or disposition not in {"retain_reference", "repair_reference", "hold_for_model_input"}
            or any(
                not isinstance(entry.get(field), str) or not entry[field].strip()
                for field in ("reviewer", "reviewer_version", "reason")
            )
        ):
            failure = "case_identity_or_disposition_invalid"
        elif disposition == "repair_reference":
            permitted = set(ref.plan.ends)
            permitted.update(item.start for item in text.paragraphs)
            permitted.update(m.end() for m in sentence_break.finditer(text.clean_text))
            permitted.update(m.end() for m in whitespace_break.finditer(text.clean_text))
            # A reviewed offset can fall inside a whitespace run or directly before a heading.
            if isinstance(ends, list):
                for end in ends:
                    if type(end) is not int or not 0 < end < len(text.clean_text):
                        continue
                    left = text.clean_text[end - 1]
                    right = text.clean_text[end]
                    whitespace_edge = (left.isspace() or right.isspace()) and (left, right) != (
                        "\r",
                        "\n",
                    )
                    before = text.clean_text[max(0, end - 4) : end].rstrip("\"'”’)】]")
                    sentence_edge = bool(
                        before
                        and (
                            (before[-1] in "。！？" and right.isalnum())
                            or (before[-1] in ".!?" and right.isupper())
                        )
                    )
                    if whitespace_edge or sentence_edge:
                        permitted.add(end)
            if not (
                isinstance(ends, list)
                and ends
                and all(type(end) is int for end in ends)
                and ends == sorted(set(ends))
                and ends[0] > 0
                and ends[-1] == len(text.clean_text)
                and set(ends) <= permitted
                and ends != ref.plan.ends
            ):
                failure = "case_repair_offsets_invalid"
        elif ends is not None:
            failure = "unexpected_offsets_for_reference_or_hold"
        elif disposition == "retain_reference":
            ends = ref.plan.ends

        counts = None
        if failure is None and ends is not None:
            counts = [
                len(encoder.encode_ordinary(text.clean_text[start:end]))
                for start, end in zip([0, *ends[:-1]], ends, strict=True)
            ]
            if any(count < 1 or count > budget for count in counts):
                failure = "case_repair_exceeds_reference_budget"
        if failure is not None:
            case_validation_failures[key] = {"batch": case_assignment[key], "failure": failure}
            continue
        case_results[key] = {
            **entry,
            "initial_status": prior[key]["status"],
            "sample": prior[key]["sample"],
            "receipt_sha256": case_receipts[case_assignment[key]],
            "reference_token_counts": counts,
        }

summary = Counter(row["disposition"] for row in case_results.values())
case_record = {
    "sample_closure_sha256": sample_closure_sha,
    "case_manifest_sha256": sha256_file(case_manifest_path),
    "source_status_counts": case_manifest["initial_status_counts"],
    "final_dispositions": dict(summary),
    "remaining_escalated": sum(
        prior[key]["status"] == "escalated" for key in pending - case_results.keys()
    ),
    "remaining_reference_budget": sum(
        prior[key]["status"] == "reference_budget_review_required"
        for key in pending - case_results.keys()
    ),
    "validation_failures": case_validation_failures,
    "decisions": list(case_results.values()),
}
with tempfile.NamedTemporaryFile(dir=RUN, mode="w", encoding="utf-8", delete=False) as handle:
    handle.write(json.dumps(case_record, indent=2) + "\n")
    staged = Path(handle.name)
staged.replace(RUN / "PRIOR_CASE_REPLAY.json")
print(f"Verified {len(case_results)} of {len(pending)} legacy blocker dispositions.")
print(
    f"Legacy blockers still open: {case_record['remaining_escalated']} semantic, "
    f"{case_record['remaining_reference_budget']} reference budget."
)
if summary:
    display(pd.DataFrame(summary.items(), columns=["Verified case disposition", "Parents"]))


## Reconcile the complete nonparagraph scope

Record all first-pass decisions and final dispositions before changing downstream eligibility.

In [ ]:
effective = {key: row["status"] for key, row in reviewed.items()}
for key, decision in case_results.items():
    if prior[key]["sample"] == "nonparagraph":
        effective[key] = (
            "hold_for_model_input"
            if decision["disposition"] == "hold_for_model_input"
            else "boundary_review_complete"
        )
effective_counts = Counter(effective.values())
new_held = [
    {
        "release_row": key,
        "body_sha256": reviewed[key]["body_sha256"],
        "review_origin": reviewed[key]["review_origin"],
        "hold_reason": status,
    }
    for key, status in sorted(effective.items())
    if key not in first_pass and status in {"escalated", "reference_budget_review_required"}
]
progress = {
    "status": (
        "full_nonparagraph_review_and_legacy_adjudication_complete"
        if len(reviewed) == risk_size and len(case_results) == len(pending)
        else "full_nonparagraph_review_in_progress"
    ),
    "review_manifest_sha256": sha256_file(manifest_path),
    "legacy_case_replay_sha256": sha256_file(RUN / "PRIOR_CASE_REPLAY.json"),
    "nonparagraph_scope": risk_size,
    "replayed_nonparagraph_parents": len(reviewed),
    "pending_first_review_parents": risk_size - len(reviewed),
    "original_case_status_counts": case_record["source_status_counts"],
    "remaining_original_escalations": case_record["remaining_escalated"],
    "remaining_original_reference_budget_cases": case_record["remaining_reference_budget"],
    "current_nonparagraph_dispositions": dict(effective_counts),
    "new_review_decision_counts": dict(replay_counts),
    "new_cases_held_from_model_input": len(new_held),
    "new_case_holds": new_held,
    "receipt_batches": sum((SHARED / a["receipt"]).is_file() for a in assignments.values()),
    "expected_review_batches": len(assignments),
    "case_receipt_batches": len(case_receipts),
    "expected_case_batches": len(case_manifest["batches"]),
    "material_nonparagraph_parents_first_pass": sum(
        "material_evidence_break" in row["evidence_outcomes"] for row in reviewed.values()
    ),
}
with tempfile.NamedTemporaryFile(dir=RUN, mode="w", encoding="utf-8", delete=False) as handle:
    handle.write(json.dumps(progress, indent=2) + "\n")
    staged = Path(handle.name)
staged.replace(RUN / "PROGRESS.json")
print(f"Replayed {len(reviewed)} of {risk_size} nonparagraph parents.")
display(pd.DataFrame(effective_counts.items(), columns=["Nonparagraph disposition", "Parents"]))
if progress["status"] == "full_nonparagraph_review_and_legacy_adjudication_complete":
    if set(reviewed) != parent_roster or set(case_results) != pending:
        raise ValueError("The full scope and legacy closure could not be reconciled.")
    target = RUN / "FULL_REVIEW_CLOSURE.json"
    with tempfile.NamedTemporaryFile(dir=RUN, mode="w", encoding="utf-8", delete=False) as handle:
        handle.write(json.dumps(progress, indent=2) + "\n")
        staged = Path(handle.name)
    staged.replace(target)


## Recheck exact cross-split chunk reuse

Apply verified offsets as a separate candidate and keep original reference holds.

In [ ]:
import pyarrow.parquet as pq

if progress["status"] == "full_nonparagraph_review_and_legacy_adjudication_complete":
    # Preserve the sampled paragraph decisions that are outside the expanded review scope.
    overrides = {
        key: row["internal_end_offsets"]
        for key, row in {**prior, **reviewed}.items()
        if row["status"] == "boundary_review_complete" and row["internal_end_offsets"] is not None
    }
    for key, adjudicated in case_results.items():
        action = adjudicated["disposition"]
        if action == "repair_reference":
            overrides[key] = adjudicated["replacement_ends"][:-1]
        else:
            overrides.pop(key, None)
    parent_ends = spark.createDataFrame(
        list(overrides.items()), "release_row long, reviewed_internal_ends array<int>"
    )
    proposed = (
        plans.join(F.broadcast(parent_ends), "release_row", "left")
        .withColumn(
            "candidate_ends",
            F.when(
                F.col("reviewed_internal_ends").isNotNull(),
                F.concat("reviewed_internal_ends", F.array("clean_chars")),
            ).otherwise(F.col("plan.ends")),
        )
        .select(
            "release_row",
            "clean_sha256",
            "ml_split",
            "candidate_ends",
            F.posexplode("candidate_ends").alias("ordinal", "end"),
        )
        .withColumn("start", F.coalesce(F.get("candidate_ends", F.col("ordinal") - 1), F.lit(0)))
        .drop("candidate_ends")
        .join(
            parents.select("release_row", "clean_sha256", "clean_text"),
            ["release_row", "clean_sha256"],
        )
        .withColumn("codepoints", F.col("end") - F.col("start"))
        .withColumn(
            "chunk_sha256", F.sha2(F.substring("clean_text", F.col("start") + 1, "codepoints"), 256)
        )
        .drop("clean_text")
        .withColumn(
            "split_mask",
            F.when(F.col("ml_split") == "train", 1)
            .when(F.col("ml_split") == "validation", 2)
            .otherwise(4),
        )
        .persist()
    )
    exact = (
        proposed.groupBy("chunk_sha256", "codepoints")
        .agg(F.bit_or("split_mask").alias("split_mask"))
        .where(F.col("split_mask").isin(3, 5, 6, 7))
        .select("chunk_sha256", "codepoints")
        .persist()
    )
    reference_holds = load_table(
        POLICY_DIR, "exact_chunk_holds", policy_manifest["tables"]["exact_chunk_holds"]
    ).select("release_row", "start", "end", "chunk_sha256")
    retained_keys = proposed.join(
        F.broadcast(reference_holds), ["release_row", "start", "end", "chunk_sha256"]
    ).select("release_row", "ordinal")
    refreshed_keys = proposed.join(F.broadcast(exact), ["chunk_sha256", "codepoints"]).select(
        "release_row", "ordinal"
    )
    all_keys = retained_keys.unionByName(refreshed_keys).distinct()
    post_hold = (
        proposed.join(F.broadcast(all_keys), ["release_row", "ordinal"], "left_anti")
        .groupBy("chunk_sha256", "codepoints")
        .agg(F.bit_or("split_mask").alias("split_mask"))
        .where(F.col("split_mask").isin(3, 5, 6, 7))
        .count()
    )
    if post_hold:
        raise ValueError("Exact cross-split reuse survived the candidate hold recheck.")
    audit = {
        "status": "full_nonparagraph_candidate_exact_recheck_complete",
        "review_closure_sha256": sha256_file(RUN / "FULL_REVIEW_CLOSURE.json"),
        "reviewed_parent_plans_used": len(overrides),
        "candidate_chunks": proposed.count(),
        "candidate_cross_split_exact_groups": exact.count(),
        "held_occurrences": all_keys.count(),
        "exact_cross_split_groups_after_holds": post_hold,
    }
    gate = RUN / "exact_candidate_gate_v1"
    if not gate.exists():
        kept = proposed.join(F.broadcast(all_keys), ["release_row", "ordinal"]).drop("split_mask")
        with tempfile.TemporaryDirectory(prefix=".candidate-holds-", dir=RUN) as temp:
            stage = Path(temp) / "snapshot"
            held_dir = stage / "held_occurrences"
            kept.write.mode("errorifexists").option("compression", "zstd").parquet(str(held_dir))
            files = [
                {
                    "path": path.relative_to(stage).as_posix(),
                    "sha256": sha256_file(path),
                    "rows": pq.ParquetFile(path).metadata.num_rows,
                }
                for path in sorted(held_dir.glob("*.parquet"))
            ]
            (stage / "MANIFEST.json").write_text(
                json.dumps({"audit": audit, "files": files}, indent=2) + "\n",
                encoding="utf-8",
            )
            if gate.exists():
                raise FileExistsError("A candidate hold snapshot appeared during validation.")
            stage.rename(gate)
    stored = json.loads((gate / "MANIFEST.json").read_text(encoding="utf-8"))
    if stored["audit"] != audit:
        raise ValueError("The saved exact hold ledger belongs to a different review closure.")
    if sum(part["rows"] for part in stored["files"]) != audit["held_occurrences"]:
        raise ValueError("The candidate exact hold row count differs from the audit.")
    load_table(gate, "held_occurrences", stored["files"])
    with tempfile.NamedTemporaryFile(dir=RUN, mode="w", encoding="utf-8", delete=False) as handle:
        handle.write(json.dumps(audit, indent=2) + "\n")
        staged = Path(handle.name)
    staged.replace(RUN / "FULL_CANDIDATE_EXACT_RECHECK.json")
    print(audit)
    proposed.unpersist()
    exact.unpersist()
